<a href="https://colab.research.google.com/github/ruicatzzz/aigc-detector/blob/daphne/notebooks/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
%cd /Users/daphnechia/Documents/GitHub/aigc-detector

# change to ur own file path

/Users/daphnechia/Documents/GitHub/aigc-detector


In [12]:
!pip install datasets --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import kagglehub
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print(path)

In [ ]:
import shutil, os

# path is whatever printed from the previous cell
os.makedirs('data/cifake', exist_ok=True)
shutil.copytree(path, 'data/cifake', dirs_exist_ok=True)

In [21]:
#load partial dataset from huggingface
from datasets import load_dataset
from pathlib import Path

N_TRAIN_USED = 10000   # however many you downloaded for training
N_TEST = 2000

out_dir = Path("data/sid_test_holdout")
(out_dir / "REAL").mkdir(parents=True, exist_ok=True)
(out_dir / "FAKE").mkdir(parents=True, exist_ok=True)

ds = load_dataset("saberzl/SID_Set", split="train", streaming=True)
ds = ds.skip(N_TRAIN_USED)  # skip past everything already used for training

real_count, fake_count = 0, 0
for i, example in enumerate(ds):
    if i >= N_TEST:
        break
    label = example["label"]
    img = example["image"]
    if label == 0:
        img.convert("RGB").save(out_dir / "REAL" / f"{example['img_id']}.jpg")
        real_count += 1
    else:
        img.convert("RGB").save(out_dir / "FAKE" / f"{example['img_id']}.jpg")
        fake_count += 1

print(f"Saved {real_count} REAL, {fake_count} FAKE held-out test images to {out_dir}")

Saved 638 REAL, 1362 FAKE held-out test images to data/sid_test_holdout


In [29]:
!python -m src.train --data_dir data/cifake/train data/sid_subset --epochs 10 --out checkpoints/cnn_merged.pt

Using device: cpu
data/cifake/train: 100000 images, classes={'FAKE': 0, 'REAL': 1}
data/sid_subset: 10000 images, classes={'FAKE': 0, 'REAL': 1}
Merged: 99000 train, 11000 val
Epoch 1/10: 100%|████████████████████████████| 774/774 [02:25<00:00,  5.31it/s]
Epoch 1: train_loss=0.4096  val_acc=0.8882  <- new best, saving
Epoch 2/10: 100%|████████████████████████████| 774/774 [03:12<00:00,  4.02it/s]
Epoch 2: train_loss=0.3162  val_acc=0.8940  <- new best, saving
Epoch 3/10: 100%|████████████████████████████| 774/774 [03:24<00:00,  3.79it/s]
Epoch 3: train_loss=0.2913  val_acc=0.9163  <- new best, saving
Epoch 4/10: 100%|████████████████████████████| 774/774 [03:34<00:00,  3.60it/s]
Epoch 4: train_loss=0.2713  val_acc=0.9166  <- new best, saving
Epoch 5/10: 100%|████████████████████████████| 774/774 [03:35<00:00,  3.59it/s]
Epoch 5: train_loss=0.2569  val_acc=0.9039
Epoch 6/10: 100%|████████████████████████████| 774/774 [03:22<00:00,  3.82it/s]
Epoch 6: train_loss=0.2418  val_acc=0.9247  <

In [17]:
#check how well the model prediction is currently

!python -m src.robustness_test


=== REAL images ===

0301 (2).jpg:
  clean              pred=0.5710
  jpeg_q30           pred=0.1450
  jpeg_q70           pred=0.7294
  blur_sigma1.0      pred=0.6107
  blur_sigma2.0      pred=0.5275
  resize_0.5x        pred=0.8525
  resize_0.25x       pred=0.6779
  noise_sigma0.05    pred=0.5729
  color_jitter       pred=0.6826
  center_crop_80     pred=0.8876

0436 (5).jpg:
  clean              pred=0.0001
  jpeg_q30           pred=0.0004
  jpeg_q70           pred=0.0002
  blur_sigma1.0      pred=0.0004
  blur_sigma2.0      pred=0.0021
  resize_0.5x        pred=0.0008
  resize_0.25x       pred=0.0014
  noise_sigma0.05    pred=0.0006
  color_jitter       pred=0.0004
  center_crop_80     pred=0.0016

0537 (8).jpg:
  clean              pred=0.1980
  jpeg_q30           pred=0.1991
  jpeg_q70           pred=0.2549
  blur_sigma1.0      pred=0.1695
  blur_sigma2.0      pred=0.2853
  resize_0.5x        pred=0.2015
  resize_0.25x       pred=0.5007
  noise_sigma0.05    pred=0.1423
  color_ji

In [30]:
# output predictions to json file

!python -m src.infer --input_dir data/cifake/test data/sid_test_holdout --output_json outputs/preds.json --checkpoint checkpoints/cnn_merged.pt

Running inference: 100%|███████████████| 22000/22000 [00:15<00:00, 1424.34it/s]
Wrote 22000 predictions to outputs/preds.json


In [31]:
#robustness summary

!python -m src.robustness_summary --csv outputs/robustness_table.csv:CIFAKE+SID_Set

Saved table to outputs/robustness_summary.md
Saved chart to outputs/robustness_chart.png
